In [1]:
!pip install gradio -q

In [2]:
import torch
import torch.nn as nn
from torchvision import models
from transformers import DistilBertTokenizer, DistilBertModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

selected_classes = [
    'Basketball', 'Biking', 'Bowling', 'CliffDiving',
    'GolfSwing', 'HorseRiding', 'Skiing', 'Surfing',
    'TennisSwing', 'SkateBoarding'
]

# ✅ Redefine model architecture
class MultimodalFusionModel(nn.Module):
    def __init__(self, num_classes=10, hidden_size=512, num_layers=2):
        super(MultimodalFusionModel, self).__init__()

        resnet = models.resnet50(pretrained=True)
        for name, param in resnet.named_parameters():
            param.requires_grad = 'layer4' in name
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])

        self.lstm = nn.LSTM(
            input_size  = 2048,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = 0.3
        )

        self.text_projector = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.fusion = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, video, text_feat):
        batch_size, frames, C, H, W = video.shape
        video = video.view(batch_size * frames, C, H, W)

        with torch.no_grad():
            cnn_out = self.cnn(video)

        cnn_out     = cnn_out.view(batch_size, frames, -1)
        lstm_out, _ = self.lstm(cnn_out)
        visual_feat = lstm_out[:, -1, :]

        text_feat = text_feat.squeeze(1) if text_feat.dim() == 3 else text_feat
        text_feat = self.text_projector(text_feat)

        fused = torch.cat([visual_feat, text_feat], dim=1)
        out   = self.fusion(fused)
        return out

# ✅ Reload DistilBERT
tokenizer  = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased')
bert_model = bert_model.to(device)
bert_model.eval()
for param in bert_model.parameters():
    param.requires_grad = False

def get_text_features(text):
    tokens = tokenizer(
        text,
        return_tensors = 'pt',
        padding        = True,
        truncation     = True,
        max_length     = 32
    ).to(device)
    with torch.no_grad():
        output = bert_model(**tokens)
    return output.last_hidden_state[:, 0, :].cpu()

# ✅ Load saved weights
checkpoint_path = '/content/drive/MyDrive/NewsImageCNN/checkpoints/fusion_best.pth'
fusion_model    = MultimodalFusionModel(num_classes=10).to(device)
fusion_model.load_state_dict(torch.load(checkpoint_path))
fusion_model.eval()

print("✅ Model loaded successfully!")
print(f"Device: {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You ca

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 175MB/s]


✅ Model loaded successfully!
Device: cuda


In [19]:
import gradio as gr
import torch
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms
import torch.nn.functional as F

checkpoint_path = '/content/drive/MyDrive/NewsImageCNN/checkpoints/fusion_best.pth'

# ✅ Make sure fusion_model is loaded
fusion_model.load_state_dict(torch.load(checkpoint_path))
fusion_model.eval()

selected_classes = [
    'Basketball', 'Biking', 'Bowling', 'CliffDiving',
    'GolfSwing', 'HorseRiding', 'Skiing', 'Surfing',
    'TennisSwing', 'SkateBoarding'
]

# Transform — same as training
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

def extract_frames_from_video(video_path, num_frames=16):
    """Extract evenly spaced frames from uploaded video"""
    cap          = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        return None

    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames  = []

    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx in indices:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = Image.fromarray(frame)
            frame = transform(frame)
            frames.append(frame)
        frame_idx += 1

    cap.release()

    if len(frames) < num_frames:
        return None

    return torch.stack(frames[:num_frames])

def predict_video(video_path, description):
    try:
        frames = extract_frames_from_video(video_path)
        if frames is None:
            return "❌ Could not extract frames", pd.DataFrame()

        video_tensor = frames.unsqueeze(0).to(device)

        if not description or description.strip() == "":
            description = "news video content"

        text_feat = get_text_features(description)
        if text_feat.dim() == 3:
            text_feat = text_feat.squeeze(1)
        text_feat = text_feat.to(device)

        with torch.no_grad():
            output = fusion_model(video_tensor, text_feat)
            probs  = F.softmax(output, dim=1)[0]

        confidence, pred_idx = torch.max(probs, 0)
        predicted_class      = selected_classes[pred_idx.item()]
        confidence_pct       = confidence.item() * 100

        # ✅ Return DataFrame for BarPlot
        df = pd.DataFrame({
            "Class"           : selected_classes,
            "Probability (%)" : [round(float(probs[i].item()) * 100, 2)
                                 for i in range(len(selected_classes))]
        })

        result = f"🎯 Predicted: {predicted_class}\n📊 Confidence: {confidence_pct:.2f}%"
        return result, df

    except Exception as e:
        return f"❌ Error: {str(e)}", pd.DataFrame()

In [20]:
import gradio as gr
import pandas as pd

with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🎬 Deep Learning News Video Content Recognition
    ### Multimodal Fusion — CNN + LSTM + DistilBERT
    Upload a video and optionally provide a text description
    to classify the content using multimodal deep learning.
    """)

    with gr.Row():

        # Left column — inputs
        with gr.Column(scale=1):
            video_input = gr.Video(
                label="📹 Upload Video"
            )
            text_input = gr.Textbox(
                label       = "📝 Video Description (optional)",
                placeholder = "e.g. person riding bicycle on road",
                lines       = 2
            )
            predict_btn = gr.Button(
                "🔍 Predict",
                variant = "primary",
                size    = "lg"
            )

        # Right column — outputs
        with gr.Column(scale=1):
            result_output = gr.Textbox(
                label       = "🎯 Prediction Result",
                lines       = 3,
                interactive = False
            )
            # ✅ BarPlot instead of Label
            prob_output = gr.BarPlot(
                x          = "Class",
                y          = "Probability (%)",
                title      = "📊 Class Probabilities",
                tooltip    = ["Class", "Probability (%)"],
                y_lim      = [0, 100],
                label      = "📊 Class Probabilities"
            )

    gr.Markdown("### 📌 Supported Classes")
    gr.Markdown("""
    `Basketball` • `Biking` • `Bowling` • `CliffDiving` • `GolfSwing`
    `HorseRiding` • `Skiing` • `Surfing` • `TennisSwing` • `SkateBoarding`
    """)

    predict_btn.click(
        fn      = predict_video,
        inputs  = [video_input, text_input],
        outputs = [result_output, prob_output]
    )

print("Gradio app ready!")

/tmp/ipykernel_3439/1204256752.py:4: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_3439/1204256752.py:4: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:


Gradio app ready!


In [21]:
demo.launch(
    share        = True,   # generates public link
    debug        = True,
    show_error   = True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e06ddb5b9e8c1a18b9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 420, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e06ddb5b9e8c1a18b9.gradio.live
